# 08 - Valid Alt Kümede Detection Screening

Bu notebook, Stage 07'den gelen sınıflandırma modellerini valid alt küme üzerinde sliding-window detector gibi değerlendirir.

Amaç yeni model eğitmek değil; Stage 09'a geçecek aday detection modellerini belirlemektir.


## 1. Kütüphaneler

Bu bölümde dosya işlemleri, tablo üretimi, görüntü işleme, model yükleme ve görselleştirme için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import time
import json
import re
import itertools
import warnings

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern

warnings.filterwarnings("ignore")

print("Kütüphaneler yüklendi.")


## 2. Ayarlar

Bu bölümde detection screening sırasında kullanılacak IoU eşikleri, seçim limitleri ve overwrite davranışı tanımlanır.


In [ ]:
RANDOM_STATE = 42

OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

IMAGE_SCALE = 1.0
POSTPROCESS_METHOD = "weighted_center"
MAX_CANDIDATES_PER_IMAGE = 500

MATCHING_IOU_THRESHOLDS = [0.10, 0.20, 0.30]
PRIMARY_MATCHING_IOU = 0.30

TOP_N_PER_GROUP = 3
ADD_BEST_NEG4X_IF_MISSING = True
MAX_MODELS_PER_GROUP = 4

ALGORITHM_DISPLAY_ORDER = ["linear_svm", "rbf_svm", "random_forest"]

THRESHOLD_COLUMN_BY_ALGORITHM = {
    "linear_svm": "score_threshold",
    "rbf_svm": "decision_function_score_threshold",
    "random_forest": "predict_proba_score_threshold",
}

print("Ayarlar yüklendi.")


## 3. Dosya Yolları

Bu bölümde önceki 08 çıktıları, Stage 07 sıralama tabloları, raw veri klasörleri ve model havuzu yolları tanımlanır.


In [ ]:
def auto_find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()
        if has_repo_layout and has_data_dir:
            return candidate
    return None


PROJECT_ROOT = auto_find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

APPROACH_ROOT = PROJECT_ROOT / "approaches" / "traditional_ml"
STAGE_NAME = "08_detection_screening"
NOTEBOOK_NAME = "01_detection_screening_on_valid_subset"

PRELIM_OUTPUT_DIR = APPROACH_ROOT / "outputs" / STAGE_NAME / "00_preliminary_detection_parameter_calibration"
PRELIM_TABLES_DIR = PRELIM_OUTPUT_DIR / "tables"

INPUT_DIR = APPROACH_ROOT / "inputs" / STAGE_NAME

STAGE_OUTPUT_DIR = APPROACH_ROOT / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = STAGE_OUTPUT_DIR / "tables"
FIGURES_DIR = STAGE_OUTPUT_DIR / "figures"

for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = {
    "dataset1_varroa": {
        "dataset_short": "dataset1",
        "dataset_index": 1,
        "varroa_class_ids": {1},
        "ranked_stage07_path": APPROACH_ROOT / "outputs" / "07_classification_final_test" / "01_dataset1_classification_final_test" / "tables" / "ranked_classification_final_test_models.csv",
        "valid_subset_path": PRELIM_TABLES_DIR / "valid_subset_selection_dataset1.csv",
        "raw_dataset_dir": PROJECT_ROOT / "data" / "raw" / "dataset1_varroa",
        "pca_artifacts_dir": PROJECT_ROOT / "data" / "features" / "dataset1_varroa" / "pca_artifacts",
        "model_pool_dir": PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool" / "dataset1_varroa",
    },
    "dataset2_honeybee": {
        "dataset_short": "dataset2",
        "dataset_index": 2,
        "varroa_class_ids": {0},
        "ranked_stage07_path": APPROACH_ROOT / "outputs" / "07_classification_final_test" / "02_dataset2_classification_final_test" / "tables" / "ranked_classification_final_test_models.csv",
        "valid_subset_path": PRELIM_TABLES_DIR / "valid_subset_selection_dataset2.csv",
        "raw_dataset_dir": PROJECT_ROOT / "data" / "raw" / "dataset2_honeybee",
        "pca_artifacts_dir": PROJECT_ROOT / "data" / "features" / "dataset2_honeybee" / "pca_artifacts",
        "model_pool_dir": PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool" / "dataset2_honeybee",
    },
}

STANDARDIZED_PARAMETERS_PATH = INPUT_DIR / "standardized_detection_parameters.csv"
SELECTED_PARAMETERS_PATH = PRELIM_TABLES_DIR / "selected_detection_parameters.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STAGE_OUTPUT_DIR:", STAGE_OUTPUT_DIR)
print("STANDARDIZED_PARAMETERS_PATH:", STANDARDIZED_PARAMETERS_PATH)

## 4. Kayıt ve Çıktı Yardımcıları

Bu bölümde tablo ve figür çıktıları için ortak kayıt fonksiyonları tanımlanır.


In [ ]:
output_log = []


def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def log_output(message):
    message = str(message)
    print(message)
    output_log.append({
        "type": "text",
        "content": message,
        "timestamp": timestamp_now(),
    })


def log_dataframe(title, df, max_rows=20):
    print(f"\n{title}")
    display(df.head(max_rows))


def save_csv(df, path, overwrite=False, note=""):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and not overwrite:
        log_output(f"[SKIP] Mevcut CSV korundu: {relative_path(path)}")
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            log_output(f"[INFO] Mevcut CSV boş olduğu için güncel tablo bellekte kullanılacak: {relative_path(path)}")
            return df

    df.to_csv(path, index=False, encoding="utf-8-sig")
    log_output(f"[SAVE] CSV kaydedildi: {relative_path(path)} | shape={df.shape}")
    return df


def save_current_figure(path, title, overwrite=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and not overwrite:
        log_output(f"[SKIP] Mevcut figür korundu: {relative_path(path)}")
        return path

    plt.savefig(path, dpi=150, bbox_inches="tight")
    log_output(f"[SAVE] Figür kaydedildi: {relative_path(path)} | {title}")
    return path



def normalize_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in ["true", "1", "yes", "y"]

print("Yardımcı fonksiyonlar hazır.")


## 5. Girdi Envanteri

Bu bölümde valid alt küme dosyaları, detection parametreleri, Stage 07 sıralamaları ve model klasörleri kontrol edilir.


In [ ]:
input_inventory_rows = []

def add_inventory_item(name, path, item_type, required=True):
    path = Path(path)
    input_inventory_rows.append({
        "name": name,
        "path": relative_path(path),
        "item_type": item_type,
        "required": bool(required),
        "exists": path.exists(),
    })

add_inventory_item("standardized_detection_parameters", STANDARDIZED_PARAMETERS_PATH, "file", required=True)
add_inventory_item("selected_detection_parameters_reference", SELECTED_PARAMETERS_PATH, "file", required=False)

for dataset_name, cfg in DATASET_CONFIGS.items():
    add_inventory_item(f"{dataset_name}_ranked_stage07", cfg["ranked_stage07_path"], "file", required=True)
    add_inventory_item(f"{dataset_name}_valid_subset", cfg["valid_subset_path"], "file", required=True)
    add_inventory_item(f"{dataset_name}_raw_dataset_dir", cfg["raw_dataset_dir"], "dir", required=True)
    add_inventory_item(f"{dataset_name}_valid_images", cfg["raw_dataset_dir"] / "valid" / "images", "dir", required=True)
    add_inventory_item(f"{dataset_name}_valid_labels", cfg["raw_dataset_dir"] / "valid" / "labels", "dir", required=True)
    add_inventory_item(f"{dataset_name}_model_pool_dir", cfg["model_pool_dir"], "dir", required=True)
    add_inventory_item(f"{dataset_name}_pca_artifacts_dir", cfg["pca_artifacts_dir"], "dir", required=True)
    for algorithm_name in ALGORITHM_DISPLAY_ORDER:
        add_inventory_item(f"{dataset_name}_{algorithm_name}_model_dir", cfg["model_pool_dir"] / algorithm_name, "dir", required=True)

input_inventory_df = pd.DataFrame(input_inventory_rows)
save_csv(input_inventory_df, TABLES_DIR / "input_inventory.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Input Inventory", input_inventory_df)

missing_required_df = input_inventory_df[(input_inventory_df["required"] == True) & (~input_inventory_df["exists"])].copy()
if len(missing_required_df) > 0:
    log_dataframe("Missing Required Inputs", missing_required_df)
    raise FileNotFoundError("Some required input files/folders are missing. See input_inventory.csv.")

log_output("[OK] Required inputs exist.")


## 6. Valid Alt Küme ve Parametreler

Bu bölümde 00 notebookundan gelen valid alt küme tabloları ve kullanılacak detection parametreleri yüklenir.


In [ ]:
valid_subset_by_dataset = {}
for dataset_name, cfg in DATASET_CONFIGS.items():
    subset_df = pd.read_csv(cfg["valid_subset_path"])
    valid_subset_by_dataset[dataset_name] = subset_df
    log_output(f"[LOAD] {dataset_name} valid subset: {subset_df.shape}")
    log_dataframe(
        f"{dataset_name} valid subset distribution",
        subset_df.groupby("varroa_count").size().reset_index(name="image_count"),
    )

standardized_params_df = pd.read_csv(STANDARDIZED_PARAMETERS_PATH)

required_param_cols = [
    "dataset_name",
    "algorithm_name",
    "threshold_column",
    "score_threshold",
    "stride_divisor",
    "stride",
    "cluster_iou_threshold",
    "weighted_box_size_factor",
]
missing_cols = [col for col in required_param_cols if col not in standardized_params_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in standardized_detection_parameters.csv: {missing_cols}")

standardized_params_df["score_threshold"] = standardized_params_df["score_threshold"].astype(float)
standardized_params_df["stride_divisor"] = standardized_params_df["stride_divisor"].astype(int)
standardized_params_df["stride"] = standardized_params_df["stride"].astype(int)
standardized_params_df["cluster_iou_threshold"] = standardized_params_df["cluster_iou_threshold"].astype(float)
standardized_params_df["weighted_box_size_factor"] = standardized_params_df["weighted_box_size_factor"].astype(float)

save_csv(
    standardized_params_df,
    TABLES_DIR / "standardized_detection_parameters_used.csv",
    overwrite=OVERWRITE_TABLES,
    note="Copy of approved standardized screening parameters used by this notebook.",
)
log_dataframe("Standardized Detection Parameters Used", standardized_params_df, max_rows=20)

log_output("[OK] Valid subsets and standardized parameters loaded.")


## 7. Screening Model Havuzu

Bu bölümde Stage 07 final test sonuçlarından detection screening için kullanılacak model havuzu oluşturulur.


In [ ]:
def infer_patch_size_from_patch_set(patch_set):
    patch_set = str(patch_set)
    match = re.search(r"centered(\d+)", patch_set)
    if match is not None:
        return int(match.group(1))

    fallback = re.search(r"(\d+)", patch_set)
    if fallback is not None:
        return int(fallback.group(1))

    raise ValueError(f"Could not infer patch size from patch_set={patch_set}")

def find_model_artifact_path(dataset_name, algorithm_name, model_file_name):
    model_dir = DATASET_CONFIGS[dataset_name]["model_pool_dir"] / algorithm_name
    direct_path = model_dir / model_file_name
    if direct_path.exists():
        return direct_path

    matches = list(model_dir.rglob(model_file_name))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise FileExistsError(f"Multiple model artifact matches found for {model_file_name}: {matches}")
    raise FileNotFoundError(f"Model artifact not found: {model_file_name} under {model_dir}")

def normalize_algorithm_name(name):
    name = str(name).strip()
    aliases = {
        "linear": "linear_svm",
        "linear_svm": "linear_svm",
        "linsvm": "linear_svm",
        "rbf": "rbf_svm",
        "rbf_svm": "rbf_svm",
        "rf": "random_forest",
        "random_forest": "random_forest",
    }
    return aliases.get(name, name)

screening_rows = []

for dataset_name, cfg in DATASET_CONFIGS.items():
    ranked_df = pd.read_csv(cfg["ranked_stage07_path"])
    ranked_df["dataset_name"] = dataset_name
    ranked_df["algorithm_name"] = ranked_df["algorithm_name"].apply(normalize_algorithm_name)

    for _, row in ranked_df.iterrows():
        algorithm_name = row["algorithm_name"]
        if algorithm_name not in ALGORITHM_DISPLAY_ORDER:
            continue

        model_file_name = str(row["model_file_name"])
        model_artifact_path = find_model_artifact_path(dataset_name, algorithm_name, model_file_name)
        patch_set = str(row["patch_set"])
        feature_set = str(row["feature_set"])
        patch_size = infer_patch_size_from_patch_set(patch_set)
        is_neg4x = "neg4x" in patch_set.lower() or "neg4x" in model_file_name.lower()

        screening_rows.append({
            "dataset_name": dataset_name,
            "dataset_short": cfg["dataset_short"],
            "algorithm_name": algorithm_name,
            "model_file_name": model_file_name,
            "model_path": relative_path(model_artifact_path),
            "patch_set": patch_set,
            "patch_size": patch_size,
            "feature_set": feature_set,
            "is_neg4x": bool(is_neg4x),
            "stage07_test_rank": row.get("test_rank", np.nan),
            "stage07_test_f1": row.get("test_f1", np.nan),
            "stage07_test_precision": row.get("test_precision", np.nan),
            "stage07_test_recall": row.get("test_recall", np.nan),
            "stage07_test_specificity": row.get("test_specificity", np.nan),
        })

screening_model_pool_df = pd.DataFrame(screening_rows)
screening_model_pool_df["is_neg4x"] = screening_model_pool_df["is_neg4x"].apply(normalize_bool)

expected_count = 54
if len(screening_model_pool_df) != expected_count:
    log_output(f"[WARNING] Expected {expected_count} models, found {len(screening_model_pool_df)}.")

save_csv(
    screening_model_pool_df,
    TABLES_DIR / "detection_screening_model_pool.csv",
    overwrite=OVERWRITE_TABLES,
    note="All Stage 07 selected models to be screened as detectors.",
)
log_dataframe("Detection Screening Model Pool", screening_model_pool_df, max_rows=20)

pool_counts_df = (
    screening_model_pool_df
    .groupby(["dataset_name", "algorithm_name", "is_neg4x"], dropna=False)
    .size()
    .reset_index(name="model_count")
)
log_dataframe("Screening Pool Counts", pool_counts_df, max_rows=50)

log_output("[OK] Screening model pool built.")


## 8. Feature Çıkarma Yardımcıları

Bu bölümde Stage 03 ile uyumlu HOG, HSV, LBP ve PCA feature çıkarma fonksiyonları tanımlanır.


In [ ]:
HOG_CONFIGS = {
    "hog8": {
        "orientations": 9,
        "pixels_per_cell": (8, 8),
        "cells_per_block": (2, 2),
        "block_norm": "L2-Hys",
        "transform_sqrt": True,
    },
    "hog16": {
        "orientations": 9,
        "pixels_per_cell": (16, 16),
        "cells_per_block": (2, 2),
        "block_norm": "L2-Hys",
        "transform_sqrt": True,
    },
}

HSV_BINS = {"h": 16, "s": 8, "v": 8}
LBP_CONFIG = {"radius": 1, "points": 8, "method": "uniform"}

pca_cache = {}

def feature_set_uses_hog(feature_set):
    feature_set = str(feature_set)
    return feature_set.startswith("hog8") or feature_set.startswith("hog16")

def get_hog_key_from_feature_set(feature_set):
    feature_set = str(feature_set)
    if feature_set.startswith("hog8"):
        return "hog8"
    if feature_set.startswith("hog16"):
        return "hog16"
    return None

def feature_set_uses_pca(feature_set):
    return "_pca" in str(feature_set)

def extract_hog_features(patch_rgb, hog_key):
    gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)
    cfg = HOG_CONFIGS[hog_key]
    features = hog(
        gray,
        orientations=cfg["orientations"],
        pixels_per_cell=cfg["pixels_per_cell"],
        cells_per_block=cfg["cells_per_block"],
        block_norm=cfg["block_norm"],
        transform_sqrt=cfg["transform_sqrt"],
        feature_vector=True,
        channel_axis=None,
    )
    return features.astype(np.float32)

def extract_hsv_features(patch_rgb):
    hsv = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2HSV)

    h_hist = cv2.calcHist([hsv], [0], None, [HSV_BINS["h"]], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [HSV_BINS["s"]], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], None, [HSV_BINS["v"]], [0, 256]).flatten()

    def normalize_hist(hist):
        hist = hist.astype(np.float32)
        total = hist.sum()
        if total > 0:
            hist = hist / total
        return hist

    return np.concatenate([
        normalize_hist(h_hist),
        normalize_hist(s_hist),
        normalize_hist(v_hist),
    ]).astype(np.float32)

def extract_lbp_features(patch_rgb):
    gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)
    radius = LBP_CONFIG["radius"]
    points = LBP_CONFIG["points"]
    method = LBP_CONFIG["method"]

    lbp = local_binary_pattern(gray, points, radius, method=method)
    n_bins = points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
    hist = hist.astype(np.float32)
    total = hist.sum()
    if total > 0:
        hist = hist / total
    return hist.astype(np.float32)

def load_pca_artifact(dataset_name, patch_set, hog_key):
    cache_key = (dataset_name, patch_set, hog_key)

    if cache_key in pca_cache:
        return pca_cache[cache_key]

    pca_artifact_path = (
        PROJECT_ROOT
        / "data"
        / "features"
        / dataset_name
        / "pca_artifacts"
        / f"{patch_set}__{hog_key}_pca.joblib"
    )

    if not pca_artifact_path.exists():
        raise FileNotFoundError(f"PCA artifact not found: {pca_artifact_path}")

    artifact = joblib.load(pca_artifact_path)

    if isinstance(artifact, dict):
        if "pca" not in artifact:
            raise KeyError(
                f"PCA artifact is a dict but does not contain key 'pca': {pca_artifact_path}. "
                f"Available keys: {list(artifact.keys())}"
            )
        pca = artifact["pca"]
    elif hasattr(artifact, "transform"):
        pca = artifact
    else:
        raise TypeError(
            f"Unsupported PCA artifact type at {pca_artifact_path}. "
            f"Expected dict with key 'pca' or sklearn-like transformer, got {type(artifact)}."
        )

    if not hasattr(pca, "transform"):
        raise TypeError(f"Loaded PCA object does not have transform(): {pca_artifact_path}")

    pca_cache[cache_key] = pca
    return pca

def build_feature_matrix_from_patches(patch_rgbs, dataset_name, patch_set, feature_set):
    feature_set = str(feature_set)

    hog_rows = []
    other_rows = []

    use_hog = feature_set_uses_hog(feature_set)
    hog_key = get_hog_key_from_feature_set(feature_set) if use_hog else None

    for patch_rgb in patch_rgbs:
        if use_hog:
            hog_rows.append(extract_hog_features(patch_rgb, hog_key))

        other_parts = []
        if "hsv" in feature_set:
            other_parts.append(extract_hsv_features(patch_rgb))
        if "lbp" in feature_set:
            other_parts.append(extract_lbp_features(patch_rgb))

        if len(other_parts) > 0:
            other_rows.append(np.concatenate(other_parts).astype(np.float32))

    if len(hog_rows) not in [0, len(patch_rgbs)]:
        raise ValueError("HOG extraction inconsistency across patches.")
    if len(other_rows) not in [0, len(patch_rgbs)]:
        raise ValueError("HSV/LBP extraction inconsistency across patches.")

    feature_parts = []

    if len(hog_rows) > 0:
        hog_matrix = np.vstack(hog_rows).astype(np.float32)
        if feature_set_uses_pca(feature_set):
            pca = load_pca_artifact(dataset_name, patch_set, hog_key)
            hog_matrix = pca.transform(hog_matrix).astype(np.float32)
        feature_parts.append(hog_matrix)

    if len(other_rows) > 0:
        other_matrix = np.vstack(other_rows).astype(np.float32)
        feature_parts.append(other_matrix)

    if len(feature_parts) == 0:
        raise ValueError(f"Unsupported feature_set: {feature_set}")

    return np.hstack(feature_parts).astype(np.float32)

print("Feature extraction helpers ready.")


## 9. Model Yükleme ve Skorlama Yardımcıları

Bu bölümde joblib model dosyaları yüklenir ve patch skorları hesaplanır.


In [ ]:
model_cache = {}

def unwrap_model_artifact(artifact):
    if isinstance(artifact, dict):
        for key in ["model", "pipeline", "estimator", "best_estimator", "clf", "classifier"]:
            if key in artifact:
                return artifact[key]
    return artifact

def load_model(model_path):
    model_path = Path(model_path)
    if model_path not in model_cache:
        artifact = joblib.load(model_path)
        model_cache[model_path] = unwrap_model_artifact(artifact)
    return model_cache[model_path]

def get_model_n_features(model):
    if hasattr(model, "n_features_in_"):
        return int(model.n_features_in_)
    if hasattr(model, "named_steps"):
        for step in model.named_steps.values():
            if hasattr(step, "n_features_in_"):
                return int(step.n_features_in_)
    return None

def score_feature_matrix(model, algorithm_name, X):
    if algorithm_name in ["linear_svm", "rbf_svm"]:
        if hasattr(model, "decision_function"):
            return np.asarray(model.decision_function(X)).ravel().astype(float)
        if hasattr(model, "predict_proba"):
            return np.asarray(model.predict_proba(X)[:, 1]).ravel().astype(float)
        return np.asarray(model.predict(X)).ravel().astype(float)

    if algorithm_name == "random_forest":
        if hasattr(model, "predict_proba"):
            return np.asarray(model.predict_proba(X)[:, 1]).ravel().astype(float)
        return np.asarray(model.predict(X)).ravel().astype(float)

    raise ValueError(f"Unsupported algorithm_name: {algorithm_name}")

print("Model loading/scoring helpers ready.")


## 10. Sliding Window Yardımcıları

Bu bölümde full image üzerinde aday pencereler üretilir ve model raw score çıktıları hazırlanır.


In [ ]:
def generate_sliding_windows(image_rgb, patch_size, stride):
    img_h, img_w = image_rgb.shape[:2]

    if img_w < patch_size or img_h < patch_size:
        return []

    x_positions = list(range(0, max(1, img_w - patch_size + 1), stride))
    y_positions = list(range(0, max(1, img_h - patch_size + 1), stride))

    last_x = img_w - patch_size
    last_y = img_h - patch_size

    if last_x >= 0 and last_x not in x_positions:
        x_positions.append(last_x)
    if last_y >= 0 and last_y not in y_positions:
        y_positions.append(last_y)

    windows = []
    for y in y_positions:
        for x in x_positions:
            windows.append({
                "x1": int(x),
                "y1": int(y),
                "x2": int(x + patch_size),
                "y2": int(y + patch_size),
            })
    return windows

def load_image_rgb(image_path):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

def compute_raw_scores_for_model(model_row, subset_df, parameter_row):
    dataset_name = model_row["dataset_name"]
    algorithm_name = model_row["algorithm_name"]
    model_path = model_row["model_path"]
    patch_set = model_row["patch_set"]
    patch_size = int(model_row["patch_size"])
    feature_set = model_row["feature_set"]

    stride_divisor = int(parameter_row["stride_divisor"])
    expected_stride = max(4, int(patch_size // stride_divisor))
    standardized_stride = int(parameter_row["stride"])

    if standardized_stride != expected_stride:
        log_output(
            f"[WARNING] Standardized stride {standardized_stride} differs from computed stride {expected_stride} "
            f"for {dataset_name}/{algorithm_name}/{patch_set}. Using computed stride."
        )

    stride = expected_stride

    model = load_model(model_path)
    start_time = time.time()

    raw_parts = []

    for img_idx, (_, image_row) in enumerate(subset_df.iterrows(), start=1):
        if img_idx == 1 or img_idx % 10 == 0:
            print(
                f"  {dataset_name} / {algorithm_name} / {model_row['model_file_name']}: "
                f"image {img_idx}/{len(subset_df)}"
            )

        image_rgb = load_image_rgb(image_row["image_path"])

        if IMAGE_SCALE != 1.0:
            raise ValueError("This notebook currently supports IMAGE_SCALE = 1.0 only.")

        windows = generate_sliding_windows(image_rgb, patch_size, stride)
        if len(windows) == 0:
            continue

        patch_rgbs = []
        metadata_rows = []

        for window_index, window in enumerate(windows):
            patch_rgb = image_rgb[window["y1"]:window["y2"], window["x1"]:window["x2"]].copy()
            patch_rgbs.append(patch_rgb)
            metadata_rows.append({
                "dataset_name": dataset_name,
                "algorithm_name": algorithm_name,
                "model_file_name": model_row["model_file_name"],
                "image_id": image_row["image_id"],
                "image_name": image_row["image_name"],
                "image_path": image_row["image_path"],
                "image_width": int(image_row["image_width"]),
                "image_height": int(image_row["image_height"]),
                "varroa_count": int(image_row["varroa_count"]),
                "gt_boxes_json": image_row["gt_boxes_json"],
                "window_index": window_index,
                "x1": window["x1"],
                "y1": window["y1"],
                "x2": window["x2"],
                "y2": window["y2"],
                "patch_size": int(patch_size),
                "stride_divisor": int(stride_divisor),
                "stride": int(stride),
                "patch_set": patch_set,
                "feature_set": feature_set,
            })

        X = build_feature_matrix_from_patches(
            patch_rgbs=patch_rgbs,
            dataset_name=dataset_name,
            patch_set=patch_set,
            feature_set=feature_set,
        )

        expected_features = get_model_n_features(model)
        if expected_features is not None and X.shape[1] != expected_features:
            raise ValueError(
                f"Feature dimension mismatch for {dataset_name}/{algorithm_name}/{feature_set}: "
                f"extracted={X.shape[1]}, model_expected={expected_features}"
            )

        scores = score_feature_matrix(model, algorithm_name, X)
        image_raw_df = pd.DataFrame(metadata_rows)
        image_raw_df["score"] = scores
        raw_parts.append(image_raw_df)

    elapsed = time.time() - start_time

    if len(raw_parts) == 0:
        combined_df = pd.DataFrame()
    else:
        combined_df = pd.concat(raw_parts, ignore_index=True)

    return combined_df, float(elapsed)

print("Sliding-window raw score helpers ready.")


## 11. Eşleştirme ve Kümeleme Yardımcıları

Bu bölümde IoU, weighted-center kümeleme ve ground-truth eşleştirme fonksiyonları tanımlanır.


In [ ]:
def compute_iou(box_a, box_b):
    if isinstance(box_a, dict):
        ax1, ay1, ax2, ay2 = box_a["x1"], box_a["y1"], box_a["x2"], box_a["y2"]
    else:
        ax1, ay1, ax2, ay2 = box_a

    if isinstance(box_b, dict):
        bx1, by1, bx2, by2 = box_b["x1"], box_b["y1"], box_b["x2"], box_b["y2"]
    else:
        bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area

    if union_area <= 0:
        return 0.0
    return float(inter_area / union_area)

def clip_box_to_image(box, image_width, image_height):
    clipped = box.copy()
    clipped["x1"] = float(max(0, min(image_width - 1, clipped["x1"])))
    clipped["y1"] = float(max(0, min(image_height - 1, clipped["y1"])))
    clipped["x2"] = float(max(0, min(image_width, clipped["x2"])))
    clipped["y2"] = float(max(0, min(image_height, clipped["y2"])))

    if clipped["x2"] < clipped["x1"]:
        clipped["x1"], clipped["x2"] = clipped["x2"], clipped["x1"]
    if clipped["y2"] < clipped["y1"]:
        clipped["y1"], clipped["y2"] = clipped["y2"], clipped["y1"]

    return clipped

def build_candidate_detections_from_raw_scores(image_raw_scores_df, score_threshold):
    candidate_df = image_raw_scores_df[image_raw_scores_df["score"] >= score_threshold].copy()

    if len(candidate_df) > MAX_CANDIDATES_PER_IMAGE:
        candidate_df = candidate_df.sort_values("score", ascending=False).head(MAX_CANDIDATES_PER_IMAGE).copy()

    detections = []
    for _, row in candidate_df.iterrows():
        detections.append({
            "x1": float(row["x1"]),
            "y1": float(row["y1"]),
            "x2": float(row["x2"]),
            "y2": float(row["y2"]),
            "score": float(row["score"]),
            "window_index": int(row["window_index"]),
        })
    return detections

def apply_weighted_center_clustering(
    detections,
    image_width,
    image_height,
    patch_size,
    score_threshold,
    cluster_iou_threshold=0.30,
    weighted_box_size_factor=1.25,
):
    if len(detections) == 0:
        return []

    remaining = sorted(detections, key=lambda d: d["score"], reverse=True)
    final_detections = []

    while len(remaining) > 0:
        seed = remaining.pop(0)
        cluster = [seed]
        new_remaining = []

        for det in remaining:
            if compute_iou(seed, det) >= cluster_iou_threshold:
                cluster.append(det)
            else:
                new_remaining.append(det)

        remaining = new_remaining

        centers_x = np.array([(d["x1"] + d["x2"]) / 2 for d in cluster], dtype=float)
        centers_y = np.array([(d["y1"] + d["y2"]) / 2 for d in cluster], dtype=float)
        scores = np.array([d["score"] for d in cluster], dtype=float)

        weights = np.maximum(scores - float(score_threshold), 0.0) + 1e-6
        weighted_cx = float(np.sum(centers_x * weights) / np.sum(weights))
        weighted_cy = float(np.sum(centers_y * weights) / np.sum(weights))

        final_size = float(patch_size) * float(weighted_box_size_factor)
        half_size = final_size / 2.0

        final_box = {
            "x1": weighted_cx - half_size,
            "y1": weighted_cy - half_size,
            "x2": weighted_cx + half_size,
            "y2": weighted_cy + half_size,
            "score": float(np.max(scores)),
            "cluster_size": int(len(cluster)),
            "cluster_mean_score": float(np.mean(scores)),
            "weighted_cx": weighted_cx,
            "weighted_cy": weighted_cy,
        }

        final_detections.append(clip_box_to_image(final_box, image_width, image_height))

    return sorted(final_detections, key=lambda d: d["score"], reverse=True)

def match_detections_to_gt(detections, gt_boxes, iou_threshold):
    gt_boxes = [box for box in gt_boxes if box.get("is_varroa", True)]

    matched_gt_indices = set()
    tp = 0
    fp = 0
    matched_ious = []

    detections_sorted = sorted(detections, key=lambda d: d["score"], reverse=True)

    for det in detections_sorted:
        best_iou = 0.0
        best_gt_idx = None

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt_indices:
                continue
            iou = compute_iou(det, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx is not None and best_iou >= iou_threshold:
            tp += 1
            matched_gt_indices.add(best_gt_idx)
            matched_ious.append(best_iou)
        else:
            fp += 1

    fn = max(0, len(gt_boxes) - len(matched_gt_indices))
    return tp, fp, fn, matched_ious

def safe_divide(numerator, denominator):
    if denominator == 0:
        return 0.0
    return float(numerator / denominator)

print("Post-processing and matching helpers ready.")


## 12. Model Değerlendirme Fonksiyonu

Bu bölümde tek bir modelin valid alt küme üzerinde detection metrikleri hesaplanır.


In [ ]:
def get_parameter_row(dataset_name, algorithm_name):
    row_df = standardized_params_df[
        (standardized_params_df["dataset_name"] == dataset_name) &
        (standardized_params_df["algorithm_name"] == algorithm_name)
    ].copy()

    if len(row_df) != 1:
        raise ValueError(
            f"Expected exactly one standardized parameter row for {dataset_name}/{algorithm_name}, "
            f"found {len(row_df)}."
        )

    return row_df.iloc[0]

def evaluate_model_from_raw_scores(model_row, raw_scores_df, parameter_row, raw_score_runtime_seconds):
    start_time = time.time()

    dataset_name = model_row["dataset_name"]
    algorithm_name = model_row["algorithm_name"]
    patch_size = int(model_row["patch_size"])

    threshold_column = parameter_row["threshold_column"]
    score_threshold = float(parameter_row["score_threshold"])
    stride_divisor = int(parameter_row["stride_divisor"])
    computed_stride = max(4, int(patch_size // stride_divisor))
    cluster_iou_threshold = float(parameter_row["cluster_iou_threshold"])
    weighted_box_size_factor = float(parameter_row["weighted_box_size_factor"])

    if len(raw_scores_df) == 0:
        raise ValueError("raw_scores_df is empty.")

    raw_stride_values = sorted(raw_scores_df["stride"].unique().tolist())
    if len(raw_stride_values) != 1 or int(raw_stride_values[0]) != computed_stride:
        raise ValueError(
            f"Stride mismatch for {model_row['model_file_name']}: "
            f"computed={computed_stride}, raw_values={raw_stride_values}"
        )

    metric_accumulator = {}
    image_level_rows = []

    for matching_iou in MATCHING_IOU_THRESHOLDS:
        metric_accumulator[matching_iou] = {
            "tp": 0,
            "fp": 0,
            "fn": 0,
            "matched_ious": [],
            "negative_image_count": 0,
            "negative_total_fp": 0,
            "negative_correct_count": 0,
        }

    total_window_count = int(len(raw_scores_df))
    total_candidate_count = 0
    total_final_detection_count = 0

    for image_name, image_raw_scores_df in raw_scores_df.groupby("image_name"):
        if image_raw_scores_df["gt_boxes_json"].nunique() != 1:
            raise ValueError(f"gt_boxes_json inconsistency for image {image_name}")

        first_row = image_raw_scores_df.iloc[0]
        image_width = int(first_row["image_width"])
        image_height = int(first_row["image_height"])
        varroa_count = int(first_row["varroa_count"])

        gt_boxes = json.loads(first_row["gt_boxes_json"])
        gt_boxes = [box for box in gt_boxes if box.get("is_varroa", True)]

        if len(gt_boxes) != varroa_count:
            raise ValueError(
                f"GT box count mismatch for image {image_name}: "
                f"varroa_count={varroa_count}, len(gt_boxes)={len(gt_boxes)}"
            )

        candidates = build_candidate_detections_from_raw_scores(
            image_raw_scores_df=image_raw_scores_df,
            score_threshold=score_threshold,
        )
        final_detections = apply_weighted_center_clustering(
            detections=candidates,
            image_width=image_width,
            image_height=image_height,
            patch_size=patch_size,
            score_threshold=score_threshold,
            cluster_iou_threshold=cluster_iou_threshold,
            weighted_box_size_factor=weighted_box_size_factor,
        )

        total_candidate_count += len(candidates)
        total_final_detection_count += len(final_detections)

        image_row_base = {
            "dataset_name": dataset_name,
            "algorithm_name": algorithm_name,
            "model_file_name": model_row["model_file_name"],
            "image_name": image_name,
            "varroa_count": varroa_count,
            "window_count": len(image_raw_scores_df),
            "candidate_count": len(candidates),
            "final_detection_count": len(final_detections),
            "threshold_column": threshold_column,
            "threshold_value": score_threshold,
            "score_threshold": score_threshold,
            "stride_divisor": stride_divisor,
            "stride": computed_stride,
            "cluster_iou_threshold": cluster_iou_threshold,
            "weighted_box_size_factor": weighted_box_size_factor,
        }

        for matching_iou in MATCHING_IOU_THRESHOLDS:
            tp, fp, fn, matched_ious = match_detections_to_gt(
                detections=final_detections,
                gt_boxes=gt_boxes,
                iou_threshold=matching_iou,
            )

            acc = metric_accumulator[matching_iou]
            acc["tp"] += tp
            acc["fp"] += fp
            acc["fn"] += fn
            acc["matched_ious"].extend(matched_ious)

            if varroa_count == 0:
                acc["negative_image_count"] += 1
                acc["negative_total_fp"] += fp
                if fp == 0:
                    acc["negative_correct_count"] += 1

            suffix = str(matching_iou).replace(".", "_")
            image_row_base[f"tp_iou_{suffix}"] = tp
            image_row_base[f"fp_iou_{suffix}"] = fp
            image_row_base[f"fn_iou_{suffix}"] = fn
            image_row_base[f"precision_iou_{suffix}"] = safe_divide(tp, tp + fp)
            image_row_base[f"recall_iou_{suffix}"] = safe_divide(tp, tp + fn)
            image_row_base[f"f1_iou_{suffix}"] = safe_divide(2 * tp, 2 * tp + fp + fn)

        image_level_rows.append(image_row_base)

    evaluation_runtime_seconds = time.time() - start_time
    image_count = int(raw_scores_df["image_name"].nunique())

    summary = {
        "dataset_name": dataset_name,
        "dataset_short": model_row["dataset_short"],
        "algorithm_name": algorithm_name,
        "model_file_name": model_row["model_file_name"],
        "model_path": model_row["model_path"],
        "patch_set": model_row["patch_set"],
        "patch_size": patch_size,
        "feature_set": model_row["feature_set"],
        "is_neg4x": bool(model_row["is_neg4x"]),
        "stage07_test_rank": model_row["stage07_test_rank"],
        "stage07_test_f1": model_row["stage07_test_f1"],
        "stage07_test_precision": model_row["stage07_test_precision"],
        "stage07_test_recall": model_row["stage07_test_recall"],
        "stage07_test_specificity": model_row["stage07_test_specificity"],
        "threshold_column": threshold_column,
        "threshold_value": score_threshold,
        "score_threshold": score_threshold,
        "stride_divisor": stride_divisor,
        "stride": computed_stride,
        "cluster_iou_threshold": cluster_iou_threshold,
        "weighted_box_size_factor": weighted_box_size_factor,
        "image_count": image_count,
        "total_window_count": total_window_count,
        "total_candidate_count": total_candidate_count,
        "total_final_detection_count": total_final_detection_count,
        "mean_candidates_per_image": safe_divide(total_candidate_count, image_count),
        "mean_final_detections_per_image": safe_divide(total_final_detection_count, image_count),
        "raw_score_runtime_seconds": float(raw_score_runtime_seconds),
        "evaluation_runtime_seconds": float(evaluation_runtime_seconds),
        "total_runtime_seconds": float(raw_score_runtime_seconds + evaluation_runtime_seconds),
    }

    for matching_iou, acc in metric_accumulator.items():
        suffix = str(matching_iou).replace(".", "_")
        tp = acc["tp"]
        fp = acc["fp"]
        fn = acc["fn"]

        summary[f"tp_iou_{suffix}"] = tp
        summary[f"fp_iou_{suffix}"] = fp
        summary[f"fn_iou_{suffix}"] = fn
        summary[f"detection_precision_iou_{suffix}"] = safe_divide(tp, tp + fp)
        summary[f"detection_recall_iou_{suffix}"] = safe_divide(tp, tp + fn)
        summary[f"detection_f1_iou_{suffix}"] = safe_divide(2 * tp, 2 * tp + fp + fn)
        summary[f"false_positives_per_image_iou_{suffix}"] = safe_divide(fp, image_count)
        summary[f"false_negatives_per_image_iou_{suffix}"] = safe_divide(fn, image_count)
        summary[f"negative_image_count_iou_{suffix}"] = acc["negative_image_count"]
        summary[f"negative_total_fp_iou_{suffix}"] = acc["negative_total_fp"]
        summary[f"negative_fp_per_negative_image_iou_{suffix}"] = safe_divide(acc["negative_total_fp"], acc["negative_image_count"])
        summary[f"negative_accuracy_iou_{suffix}"] = safe_divide(acc["negative_correct_count"], acc["negative_image_count"])
        summary[f"mean_matched_iou_iou_{suffix}"] = float(np.mean(acc["matched_ious"])) if len(acc["matched_ious"]) > 0 else 0.0

    image_level_df = pd.DataFrame(image_level_rows)
    return summary, image_level_df

print("Model evaluation helper ready.")


## 13. Detection Screening

Bu bölümde model havuzundaki modeller valid alt küme üzerinde detector gibi değerlendirilir.


In [ ]:
summary_rows = []
image_level_result_parts = []

screening_start_time = time.time()

for model_idx, model_row in screening_model_pool_df.reset_index(drop=True).iterrows():
    dataset_name = model_row["dataset_name"]
    algorithm_name = model_row["algorithm_name"]
    subset_df = valid_subset_by_dataset[dataset_name].reset_index(drop=True)
    parameter_row = get_parameter_row(dataset_name, algorithm_name)

    log_output("=" * 100)
    log_output(
        f"[MODEL {model_idx + 1}/{len(screening_model_pool_df)}] "
        f"{dataset_name} / {algorithm_name} / {model_row['model_file_name']}"
    )

    raw_scores_df, raw_score_runtime_seconds = compute_raw_scores_for_model(
        model_row=model_row,
        subset_df=subset_df,
        parameter_row=parameter_row,
    )

    log_output(
        f"[RAW SCORE DONE] rows={len(raw_scores_df)} | runtime={raw_score_runtime_seconds:.2f}s"
    )

    summary, image_level_df = evaluate_model_from_raw_scores(
        model_row=model_row,
        raw_scores_df=raw_scores_df,
        parameter_row=parameter_row,
        raw_score_runtime_seconds=raw_score_runtime_seconds,
    )

    summary_rows.append(summary)
    image_level_result_parts.append(image_level_df)

    primary_suffix = str(PRIMARY_MATCHING_IOU).replace(".", "_")
    log_output(
        f"[MODEL DONE] F1@{PRIMARY_MATCHING_IOU}={summary[f'detection_f1_iou_{primary_suffix}']:.4f} | "
        f"TP={summary[f'tp_iou_{primary_suffix}']} "
        f"FP={summary[f'fp_iou_{primary_suffix}']} "
        f"FN={summary[f'fn_iou_{primary_suffix}']}"
    )

total_screening_runtime = time.time() - screening_start_time

detection_screening_results_df = pd.DataFrame(summary_rows)
detection_screening_image_level_results_df = pd.concat(image_level_result_parts, ignore_index=True)

save_csv(
    detection_screening_results_df,
    TABLES_DIR / "detection_screening_results.csv",
    overwrite=OVERWRITE_TABLES,
    note="Model-level detection screening results for all selected models.",
)
save_csv(
    detection_screening_image_level_results_df,
    TABLES_DIR / "detection_screening_image_level_results.csv",
    overwrite=OVERWRITE_TABLES,
    note="Image-level detection screening results.",
)

log_dataframe("Detection Screening Results Preview", detection_screening_results_df, max_rows=20)
log_output(f"[OK] Detection screening completed in {total_screening_runtime:.2f} seconds.")


## 14. Sıralama ve Aday Seçimi

Bu bölümde screening sonuçları sıralanır ve Stage 09 için aday model listesi oluşturulur.


In [ ]:
primary_suffix = str(PRIMARY_MATCHING_IOU).replace(".", "_")

if 0.20 in MATCHING_IOU_THRESHOLDS:
    secondary_iou = 0.20
else:
    secondary_candidates = [iou for iou in MATCHING_IOU_THRESHOLDS if iou != PRIMARY_MATCHING_IOU]
    secondary_iou = secondary_candidates[-1] if len(secondary_candidates) > 0 else PRIMARY_MATCHING_IOU

secondary_suffix = str(secondary_iou).replace(".", "_")

rank_columns = [
    f"detection_f1_iou_{primary_suffix}",
    f"false_positives_per_image_iou_{primary_suffix}",
    f"detection_recall_iou_{primary_suffix}",
    f"detection_f1_iou_{secondary_suffix}",
    "mean_final_detections_per_image",
    "total_runtime_seconds",
]
ascending_values = [
    False,
    True,
    False,
    False,
    True,
    True,
]

ranked_parts = []
selected_parts = []

for (dataset_name, algorithm_name), group_df in detection_screening_results_df.groupby(["dataset_name", "algorithm_name"]):
    group_ranked = group_df.sort_values(
        by=rank_columns,
        ascending=ascending_values,
    ).reset_index(drop=True)

    group_ranked["group_rank"] = np.arange(1, len(group_ranked) + 1)
    group_ranked["is_neg4x"] = group_ranked["is_neg4x"].apply(normalize_bool)
    ranked_parts.append(group_ranked)

    selected_df = group_ranked.head(TOP_N_PER_GROUP).copy()
    selected_df["selection_source"] = f"top_{TOP_N_PER_GROUP}"

    if ADD_BEST_NEG4X_IF_MISSING and not selected_df["is_neg4x"].any():
        best_neg4x_df = group_ranked[group_ranked["is_neg4x"] == True].head(1).copy()
        if len(best_neg4x_df) > 0:
            best_neg4x_df["selection_source"] = "best_neg4x_added"
            selected_df = pd.concat([selected_df, best_neg4x_df], ignore_index=True)

    selected_df = selected_df.head(MAX_MODELS_PER_GROUP).copy()
    selected_df["selected_for_stage09"] = True
    selected_parts.append(selected_df)

ranked_detection_screening_results_df = pd.concat(ranked_parts, ignore_index=True)
selected_detection_screening_models_df = pd.concat(selected_parts, ignore_index=True)

selected_detection_screening_models_df = selected_detection_screening_models_df.sort_values(
    ["dataset_name", "algorithm_name", "group_rank"]
).reset_index(drop=True)

save_csv(
    ranked_detection_screening_results_df,
    TABLES_DIR / "ranked_detection_screening_results.csv",
    overwrite=OVERWRITE_TABLES,
    note="Ranked detection screening results by dataset × algorithm group.",
)
save_csv(
    selected_detection_screening_models_df,
    TABLES_DIR / "selected_detection_screening_models.csv",
    overwrite=OVERWRITE_TABLES,
    note="Max-24 Stage 09 candidate detection models selected from valid subset screening.",
)

log_dataframe("Selected Detection Screening Models", selected_detection_screening_models_df, max_rows=30)

selection_counts_df = (
    selected_detection_screening_models_df
    .groupby(["dataset_name", "algorithm_name", "selection_source"], dropna=False)
    .size()
    .reset_index(name="selected_model_count")
)
log_dataframe("Selection Counts", selection_counts_df, max_rows=50)


## 15. Kalite Kontrolü

Bu bölümde aday üretmeyen, sıfır F1 veren veya Stage 09 seçimi olmayan gruplar kontrol edilir.


In [ ]:
quality_rows = []

for (dataset_name, algorithm_name), group_df in detection_screening_results_df.groupby(["dataset_name", "algorithm_name"]):
    primary_f1_col = f"detection_f1_iou_{primary_suffix}"
    total_candidate_sum = int(group_df["total_candidate_count"].sum())
    best_f1 = float(group_df[primary_f1_col].max())
    model_count = int(len(group_df))
    selected_count = int(
        len(
            selected_detection_screening_models_df[
                (selected_detection_screening_models_df["dataset_name"] == dataset_name) &
                (selected_detection_screening_models_df["algorithm_name"] == algorithm_name)
            ]
        )
    )

    warning_messages = []
    if total_candidate_sum == 0:
        warning_messages.append("No candidates generated by any model in this group.")
    if best_f1 == 0:
        warning_messages.append(f"Best detection F1 at IoU@{PRIMARY_MATCHING_IOU} is zero.")
    if selected_count == 0:
        warning_messages.append("No model selected for Stage 09.")

    quality_rows.append({
        "dataset_name": dataset_name,
        "algorithm_name": algorithm_name,
        "model_count": model_count,
        "selected_count": selected_count,
        "total_candidate_count": total_candidate_sum,
        f"best_detection_f1_iou_{primary_suffix}": best_f1,
        "warning_count": len(warning_messages),
        "warning_messages": " | ".join(warning_messages),
        "quality_status": "OK" if len(warning_messages) == 0 else "WARNING",
    })

screening_quality_check_df = pd.DataFrame(quality_rows)
save_csv(screening_quality_check_df, TABLES_DIR / "screening_quality_check.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Screening Quality Check", screening_quality_check_df, max_rows=50)

quality_warning_count = int((screening_quality_check_df["quality_status"] != "OK").sum())
if quality_warning_count > 0:
    log_output(f"[WARNING] Screening quality warnings found: {quality_warning_count}")
else:
    log_output("[OK] Screening quality checks passed.")


## 16. Görseller

Bu bölümde en iyi grup sonuçları ve seçilen aday modeller görselleştirilir.


In [ ]:
plot_df = ranked_detection_screening_results_df.copy()
plot_df["group_label"] = plot_df["dataset_name"] + " / " + plot_df["algorithm_name"]

# Figure 1: Top F1 by group
best_by_group_df = (
    plot_df
    .sort_values(f"detection_f1_iou_{primary_suffix}", ascending=False)
    .groupby(["dataset_name", "algorithm_name"], as_index=False)
    .head(1)
    .copy()
)
best_by_group_df["group_label"] = best_by_group_df["dataset_name"] + " / " + best_by_group_df["algorithm_name"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(best_by_group_df["group_label"], best_by_group_df[f"detection_f1_iou_{primary_suffix}"])
ax.set_title(f"Best Detection Screening F1 by Group — IoU@{PRIMARY_MATCHING_IOU}")
ax.set_xlabel("Dataset / Algorithm")
ax.set_ylabel("Best Detection F1")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig_path = FIGURES_DIR / "best_detection_screening_f1_by_group.png"
save_current_figure(fig_path, "Detection screening figure", overwrite=OVERWRITE_FIGURES)
plt.show()
# Figure path logged by save_current_figure.

# Figure 2: Selected model F1 distribution
selected_plot_df = selected_detection_screening_models_df.copy()
selected_plot_df["group_label"] = selected_plot_df["dataset_name"] + " / " + selected_plot_df["algorithm_name"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(selected_plot_df["model_file_name"], selected_plot_df[f"detection_f1_iou_{primary_suffix}"])
ax.set_title(f"Selected Stage 09 Candidate Model F1 — IoU@{PRIMARY_MATCHING_IOU}")
ax.set_xlabel("Selected model")
ax.set_ylabel("Detection F1")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
fig_path = FIGURES_DIR / "selected_stage09_candidate_f1.png"
save_current_figure(fig_path, "Detection screening figure", overwrite=OVERWRITE_FIGURES)
plt.show()
# Figure path logged by save_current_figure.


## 17. Final Durum

Bu bölümde notebook çıktılarının kısa durum özeti kaydedilir.


In [ ]:
status_value = "READY_FOR_NEXT_STAGE" if quality_warning_count == 0 else "COMPLETED_WITH_WARNINGS"

status_rows = []

for dataset_name, subset_df in valid_subset_by_dataset.items():
    status_rows.append({
        "item": f"{dataset_name}_valid_subset_image_count",
        "value": len(subset_df),
    })

status_rows.extend([
    {"item": "screened_model_count", "value": len(detection_screening_results_df)},
    {"item": "selected_stage09_candidate_count", "value": len(selected_detection_screening_models_df)},
    {"item": "matching_iou_thresholds", "value": json.dumps(MATCHING_IOU_THRESHOLDS)},
    {"item": "primary_matching_iou", "value": PRIMARY_MATCHING_IOU},
    {"item": "secondary_iou_for_tie_break", "value": secondary_iou},
    {"item": "postprocess_method", "value": POSTPROCESS_METHOD},
    {"item": "image_scale", "value": IMAGE_SCALE},
    {"item": "max_candidates_per_image", "value": MAX_CANDIDATES_PER_IMAGE},
    {"item": "standardized_parameters_path", "value": relative_path(STANDARDIZED_PARAMETERS_PATH)},
    {"item": "total_screening_runtime_seconds", "value": round(total_screening_runtime, 3)},
    {"item": "quality_warning_count", "value": quality_warning_count},
    {"item": "status", "value": status_value},
])

notebook_status_df = pd.DataFrame(status_rows)
save_csv(notebook_status_df, TABLES_DIR / "notebook_status.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Notebook Status", notebook_status_df, max_rows=50)

log_output(f"[STATUS] 01_detection_screening_on_valid_subset: {status_value}")


## 18. Stage 09 Aday Listesi

Bu bölümde manuel gözden geçirilmiş Stage 09 aday model listesi tablo olarak kaydedilir.


In [ ]:
# ------------------------------------------------------------
# Manual-reviewed Stage 09 candidate model list
# ------------------------------------------------------------

STAGE09_CANDIDATE_MODEL_FILES = [
    # Dataset1 — Linear SVM
    "d1_linsvm_011__centered80_balanced__hog16_pca_hsv_lbp.joblib",
    "d1_linsvm_004__centered80_balanced__hog16_hsv_lbp.joblib",
    "d1_linsvm_037__centered96_balanced__hog16_pca_hsv_lbp.joblib",
    "d1_linsvm_024__centered80_neg4x__hog16_pca_hsv_lbp.joblib",

    # Dataset1 — RBF SVM
    "d1_rbfsvm_007__centered80_neg4x__hog16_pca_hsv_lbp.joblib",
    "d1_rbfsvm_006__centered80_neg4x__hog16_hsv_lbp.joblib",
    "d1_rbfsvm_005__centered80_balanced__hog8_pca_hsv_lbp.joblib",
    "d1_rbfsvm_002__centered96_balanced__hog16_pca_hsv_lbp.joblib",

    # Dataset1 — Random Forest
    "d1_rf_006__centered80_neg4x__hog16_hsv_lbp.joblib",
    "d1_rf_007__centered80_neg4x__hog16_pca_hsv_lbp.joblib",
    "d1_rf_009__centered80_balanced__hsv_lbp_only.joblib",
    "d1_rf_001__centered80_balanced__hog16_pca_hsv_lbp.joblib",

    # Dataset2 — Linear SVM
    "d2_linsvm_011__centered48_balanced__hog16_pca_hsv_lbp.joblib",
    "d2_linsvm_004__centered48_balanced__hog16_hsv_lbp.joblib",
    "d2_linsvm_008__centered48_balanced__hog8_pca_hsv_lbp.joblib",
    "d2_linsvm_024__centered48_neg4x__hog16_pca_hsv_lbp.joblib",

    # Dataset2 — RBF SVM
    "d2_rbfsvm_007__centered48_neg4x__hog16_pca_hsv_lbp.joblib",
    "d2_rbfsvm_006__centered48_neg4x__hog16_hsv_lbp.joblib",
    "d2_rbfsvm_003__centered48_balanced__hog16_pca_hsv_lbp.joblib",
    "d2_rbfsvm_009__centered48_balanced__hog8_pca_hsv_lbp.joblib",

    # Dataset2 — Random Forest
    "d2_rf_007__centered48_neg4x__hog16_pca_hsv_lbp.joblib",
    "d2_rf_006__centered48_neg4x__hog16_hsv_lbp.joblib",
    "d2_rf_003__centered48_balanced__hog16_pca_hsv_lbp.joblib",
    "d2_rf_009__centered48_balanced__hog8_pca_hsv_lbp.joblib",
]

# Use the full screening results table produced by this notebook.
candidate_df = detection_screening_results_df[
    detection_screening_results_df["model_file_name"].isin(STAGE09_CANDIDATE_MODEL_FILES)
].copy()

# Safety checks.
missing_models = sorted(set(STAGE09_CANDIDATE_MODEL_FILES) - set(candidate_df["model_file_name"]))
extra_models = sorted(set(candidate_df["model_file_name"]) - set(STAGE09_CANDIDATE_MODEL_FILES))

if missing_models:
    raise ValueError(f"Missing Stage 09 candidate models in screening results: {missing_models}")

if extra_models:
    raise ValueError(f"Unexpected extra models selected: {extra_models}")

if len(candidate_df) != len(STAGE09_CANDIDATE_MODEL_FILES):
    raise ValueError(
        f"Expected {len(STAGE09_CANDIDATE_MODEL_FILES)} Stage 09 candidates, got {len(candidate_df)}."
    )

# Preserve manual order.
candidate_order_df = pd.DataFrame({
    "model_file_name": STAGE09_CANDIDATE_MODEL_FILES,
    "manual_stage09_order": range(1, len(STAGE09_CANDIDATE_MODEL_FILES) + 1),
})

candidate_df = candidate_df.merge(candidate_order_df, on="model_file_name", how="left")
candidate_df = candidate_df.sort_values("manual_stage09_order").reset_index(drop=True)

candidate_df["selection_type"] = "manual_reviewed_stage09_candidate"
candidate_df["selection_reason"] = (
    "Selected after manual review of Stage 08 valid-subset detection screening. "
    "The final pool keeps four models per dataset × algorithm group where possible, "
    "preserving detection rank, neg4x/balanced diversity, and feature-set diversity."
)

# Optional: add group candidate rank.
candidate_df["stage09_group_candidate_rank"] = (
    candidate_df
    .groupby(["dataset_name", "algorithm_name"])
    .cumcount() + 1
)

stage09_candidate_path = TABLES_DIR / "stage09_candidate_models.csv"

save_csv(
    candidate_df,
    stage09_candidate_path,
    overwrite=OVERWRITE_TABLES,
    note="Manual-reviewed final Stage 09 candidate model pool selected from Stage 08 detection screening.",
)

log_dataframe("Stage 09 Candidate Models", candidate_df, max_rows=30)

log_output(
    f"[OK] Stage 09 candidate model pool saved: {stage09_candidate_path} | "
    f"model_count={len(candidate_df)}"
)
